# Practice: Building a QA Test Suite for a Tumor Classifier

**Dataset:** [Wisconsin Diagnostic Breast Cancer dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) (`sklearn.datasets.load_breast_cancer`) — 569 real biopsy samples, 30 numeric features computed from digitized cell-nuclei images, label `malignant` (0) / `benign` (1).

**Problem statement.** You've joined an ML team as the embedded QA engineer. They've trained a logistic regression model that flags biopsies as malignant or benign, and they want to ship it. Your job is not to retrain the model — it's to build the automated test suite that decides whether *any* version of this model, now or after the next retrain, is safe to deploy. A silent regression here means a missed cancer diagnosis, so "it scored well on accuracy" is not good enough: you need checks that would actually catch the failure modes that matter.

**What you should produce**, as working Python functions (not prose):

1. `validate_schema(df, expected_columns)` and `validate_ranges(df, bounds)` — data validation checks that catch malformed input *before* it reaches the model.
2. `evaluate_model_quality(y_true, y_pred, min_recall, positive_label)` — a quality gate that fails the build if **recall on the malignant class** drops below a threshold (a missed malignant case is far more costly than a false alarm — accuracy alone would hide that).
3. `detect_drift(reference_df, current_df, alpha)` — per-feature covariate drift detection between the training distribution and a new batch, using the two-sample Kolmogorov–Smirnov test.
4. `check_reproducibility(model_fn, X_train, y_train, X_eval, seed)` — a sanity check that the *same* seed gives identical predictions and a *different* seed is allowed to give different ones (if it can't, your "reproducibility" check is meaningless).
5. A short QA report that runs all four checks and prints PASS/FAIL/FLAG for each, on both a clean batch and a batch you've deliberately drifted.

Work through the sections in order — each builds on data/variables from the one before it.

## Setup

Run this cell as-is. It loads the real dataset, builds a labeled dataframe, and creates the train/test split every later section uses.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score

data = load_breast_cancer(as_frame=True)
df = data.frame.rename(columns={"target": "label"})  # label: 0 = malignant, 1 = benign
feature_cols = [c for c in df.columns if c != "label"]

X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

print(df.shape, df["label"].value_counts().to_dict())
df.head()

## Part 1 — Data validation tests

Before any model sees new data, the pipeline should reject malformed rows: missing columns, nulls, or values outside a sane range (a negative "mean radius" is a bug, not a biopsy).

Implement:
- `validate_schema(df, expected_columns)` — returns a list of violation strings (empty list = pass). Check for missing columns and null values in the expected columns.
- `validate_ranges(df, bounds)` — `bounds` is `{column: (lo, hi)}`. Returns a list of violation strings for any column with values outside its `[lo, hi]` range.

In [ ]:
def validate_schema(df, expected_columns):
    # Return a list of violation strings; an empty list means the check passed.
    violations = []
    # TODO: check for columns in expected_columns missing from df
    # TODO: check for null values in the expected columns that ARE present
    return violations


def validate_ranges(df, bounds):
    # bounds: {column_name: (lo, hi)}. Return a list of violation strings.
    violations = []
    # TODO: for each (column, (lo, hi)) in bounds, flag rows outside [lo, hi]
    return violations


# Build a reasonable bounds dict to test against, then run both checks.
# TODO: construct `bounds` from the training data (e.g. [0, 1.5x the observed max] per column)
bounds = {}

# TODO: call validate_schema and validate_ranges on `df` and print the results


## Part 2 — A model quality gate (not just accuracy)

Train the model below (given), then implement a quality gate that checks **recall on the malignant class specifically**. In this dataset `label == 0` is malignant. A model that's 97% accurate but misses 1 in 5 malignant cases is not shippable — your gate must catch that, not just report an overall accuracy number.

In [ ]:
scaler = StandardScaler().fit(X_train)
clf = LogisticRegression(max_iter=5000, random_state=42)
clf.fit(scaler.transform(X_train), y_train)
y_pred = clf.predict(scaler.transform(X_test))


def evaluate_model_quality(y_true, y_pred, min_recall=0.95, positive_label=0):
    # Return a dict of accuracy/precision/recall/f1 for `positive_label`, plus a `passed` flag.
    # TODO: compute accuracy, and precision/recall/f1 for `positive_label` (malignant = 0)
    # TODO: set passed = (recall >= min_recall)
    # TODO: return a dict with all of the above plus min_recall_required and passed
    pass


# TODO: call evaluate_model_quality(y_test, y_pred, min_recall=0.95, positive_label=0) and print it


## Part 3 — Drift detection

Even a model that passed its quality gate at launch can degrade silently if the *input* distribution shifts in production (new imaging equipment, a different patient population, a data pipeline bug upstream). You detect that by comparing the distribution of each feature in a new batch against the training distribution with a statistical test — here, the two-sample Kolmogorov–Smirnov (KS) test, which flags a feature as drifted when its p-value falls below `alpha`.

**Why a synthetic shift here:** real production drift isn't available in a static dataset, so this cell deliberately *injects* a shift into a copy of the test batch for 3 features, to give you a case with a known, controllable answer to validate your `detect_drift` function against (alongside an undrifted batch, which should trigger no flags).

In [ ]:
rng = np.random.RandomState(0)
clean_batch = X_test.copy()                    # no drift — should trigger ~0 flagged features
drifted_batch = X_test.copy()                   # synthetic shift injected on purpose, see markdown above
for c in ["mean radius", "mean texture", "mean area"]:
    drifted_batch[c] = drifted_batch[c] + rng.normal(
        loc=drifted_batch[c].std() * 1.5, scale=0.01, size=len(drifted_batch)
    )


def detect_drift(reference_df, current_df, alpha=0.05):
    # Return a DataFrame with columns: feature, ks_stat, p_value, drifted.
    rows = []
    # TODO: for each shared column, run scipy.stats.ks_2samp(reference_df[col], current_df[col])
    # TODO: append a row dict with feature/ks_stat/p_value/drifted (p_value < alpha)
    return pd.DataFrame(rows)


# TODO: run detect_drift(X_train, clean_batch) and detect_drift(X_train, drifted_batch)
# TODO: print how many features were flagged as drifted in each case


## Part 4 — Reproducibility check

A "reproducible" training pipeline should give identical predictions when re-run with the same random seed — and it's fair for it to give *different* predictions with a different seed (otherwise the seed isn't doing anything, and your check would pass even on a broken, non-deterministic pipeline). Use a shallow, small `RandomForestClassifier` (seed genuinely affects its bootstrap sampling, unlike a converged logistic regression) and compare predicted probabilities rather than hard labels, for a more sensitive check.

In [ ]:
def check_reproducibility(model_fn, X_train, y_train, X_eval, seed=42):
    # model_fn(seed) -> an unfitted sklearn estimator.
    # Return a dict with two booleans: same_seed_reproducible and different_seed_can_differ.
    # TODO: fit model_fn(seed) twice, compare predict_proba(X_eval) for exact equality -> same_seed_reproducible
    # TODO: fit model_fn(seed + 1) once, compare predict_proba(X_eval) against the seed run -> different_seed_can_differ
    pass


# TODO: call check_reproducibility with model_fn = lambda s: RandomForestClassifier(n_estimators=20, max_depth=3, random_state=s)
# TODO: print the result


## Part 5 — Put it together: the QA report

Write a function that runs all four checks and prints a one-line PASS/FAIL/FLAG summary for each — this is the thing that would actually run in CI before a deploy.

In [ ]:
def run_qa_report():
    # TODO: run validate_schema + validate_ranges on df, print PASS/FAIL
    # TODO: run evaluate_model_quality on (y_test, y_pred), print PASS/FAIL with the recall value
    # TODO: run detect_drift on (X_train, clean_batch) and (X_train, drifted_batch), print flagged counts
    # TODO: run check_reproducibility, print PASS/FAIL
    pass


run_qa_report()


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def validate_schema(df, expected_columns):
    violations = []
    missing = set(expected_columns) - set(df.columns)
    if missing:
        violations.append(f"missing columns: {sorted(missing)}")
    present = [c for c in expected_columns if c in df.columns]
    null_cols = df[present].columns[df[present].isnull().any()].tolist()
    if null_cols:
        violations.append(f"null values in columns: {null_cols}")
    return violations


def validate_ranges(df, bounds):
    violations = []
    for col, (lo, hi) in bounds.items():
        if col not in df.columns:
            continue
        out_of_range = ~df[col].between(lo, hi)
        if out_of_range.any():
            violations.append(f"{col}: {int(out_of_range.sum())} rows out of [{lo:.3f}, {hi:.3f}]")
    return violations


bounds = {c: (0.0, df[c].max() * 1.5) for c in feature_cols}
print("schema:", validate_schema(df, feature_cols + ["label"]))
print("ranges:", validate_ranges(df, bounds))


def evaluate_model_quality(y_true, y_pred, min_recall=0.95, positive_label=0):
    recall = recall_score(y_true, y_pred, pos_label=positive_label)
    precision = precision_score(y_true, y_pred, pos_label=positive_label)
    f1 = f1_score(y_true, y_pred, pos_label=positive_label)
    acc = accuracy_score(y_true, y_pred)
    return {
        "accuracy": acc, "precision": precision, "recall": recall, "f1": f1,
        "min_recall_required": min_recall, "passed": recall >= min_recall,
    }

quality = evaluate_model_quality(y_test, y_pred, min_recall=0.95, positive_label=0)
print(quality)


def detect_drift(reference_df, current_df, alpha=0.05):
    rows = []
    for col in reference_df.columns:
        stat, p_value = stats.ks_2samp(reference_df[col], current_df[col])
        rows.append({"feature": col, "ks_stat": stat, "p_value": p_value, "drifted": p_value < alpha})
    return pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)

clean_report = detect_drift(X_train, clean_batch)
shifted_report = detect_drift(X_train, drifted_batch)
print("clean flagged:", clean_report["drifted"].sum())
print("shifted flagged:", shifted_report["drifted"].sum())


def check_reproducibility(model_fn, X_train, y_train, X_eval, seed=42):
    m1 = model_fn(seed); m1.fit(X_train, y_train)
    m2 = model_fn(seed); m2.fit(X_train, y_train)
    same_seed_reproducible = np.array_equal(m1.predict_proba(X_eval), m2.predict_proba(X_eval))
    m3 = model_fn(seed + 1); m3.fit(X_train, y_train)
    different_seed_can_differ = not np.array_equal(m1.predict_proba(X_eval), m3.predict_proba(X_eval))
    return {"same_seed_reproducible": same_seed_reproducible, "different_seed_can_differ": different_seed_can_differ}

repro = check_reproducibility(
    lambda s: RandomForestClassifier(n_estimators=20, max_depth=3, random_state=s),
    X_train, y_train, X_test, seed=42,
)
print(repro)


def run_qa_report():
    schema_violations = validate_schema(df, feature_cols + ["label"]) + validate_ranges(df, bounds)
    print("[%s] data validation" % ("PASS" if not schema_violations else "FAIL"))

    q = evaluate_model_quality(y_test, y_pred, min_recall=0.95, positive_label=0)
    print("[%s] model quality gate (recall=%.3f)" % ("PASS" if q["passed"] else "FAIL", q["recall"]))

    c_report, s_report = detect_drift(X_train, clean_batch), detect_drift(X_train, drifted_batch)
    print("[%s] drift: clean batch, %d features flagged" % (
        "PASS" if c_report["drifted"].sum() == 0 else "FLAG", c_report["drifted"].sum()))
    print("[FLAG] drift: shifted batch, %d features flagged" % s_report["drifted"].sum())

    r = check_reproducibility(
        lambda s: RandomForestClassifier(n_estimators=20, max_depth=3, random_state=s),
        X_train, y_train, X_test, seed=42,
    )
    print("[%s] reproducibility" % ("PASS" if r["same_seed_reproducible"] else "FAIL"), r)

run_qa_report()
```

**Why this matters:** each check targets a distinct, real failure mode — malformed input (schema/range), a model that looks good on an aggregate metric but fails the costly class (recall gate), silent production drift (KS test), and a non-deterministic "reproducible" pipeline (the seed-difference check, which most reproducibility tests skip — if different seeds can never produce different output, the same-seed check is vacuous). Note the deliberately injected shift in Part 3 reliably flags exactly the 3 shifted features and none of the other 27, which is what makes it useful as a test case for `detect_drift` itself.

</details>